In [1]:
from Models.modl import modl
from torch.utils.data import DataLoader
from Transforms import (pad, trim_coils, combine_coil, toTensor, permute, 
                        view_as_real, remove_slice_dim, fft_2d, normalize, addChannels)
from Dataset.undersampled_dataset import UndersampledKSpaceDataset
from torchvision.transforms import Compose
import numpy as np
import torch
from Models.varnet import VarNet

In [2]:
def collate_fn(data):
    undersampled = [d['undersampled'] for d in data]
    sampled = [d['k_space'] for d in data]
    ismrmrd_header = [d['ismrmrd_header'] for d in data]
    mask = [d['mask'] for d in data]
    recon_rss = [d['reconstruction_rss'] for d in data]

    undersampled = torch.concat(undersampled, dim=0)
    sampled = torch.concat(sampled, dim=0)
    mask = torch.concat(mask, dim=0)

    data = {
        'undersampled': undersampled, 
        'sampled': sampled,
        'ismrmrd_header': ismrmrd_header,
        'mask': mask, 
        'recon': recon_rss,
    }
    return data

In [3]:
transforms = Compose(
    (
        pad((640, 320)), 
        toTensor(),
        normalize(),
    )
)
dataset = UndersampledKSpaceDataset('/home/kadotab/projects/def-mchiew/kadotab/dataset/multicoil_train', transforms=transforms)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    

In [4]:
data = next(iter(dataloader))

In [5]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [6]:
model = VarNet(2, 2)
model.to(device)

VarNet(
  (cascade): ModuleList(
    (0): VarnetBlock(
      (unet): Unet(
        (down_sample_layers): ModuleList(
          (0): double_conv(
            (conv1): Conv2d(2, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (conv2): Conv2d(18, 18, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (activation): LeakyReLU(negative_slope=0.2, inplace=True)
            (instance_norm1): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (instance_norm2): InstanceNorm2d(18, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (drop_out1): Dropout2d(p=0, inplace=False)
            (drop_out2): Dropout2d(p=0, inplace=False)
          )
          (1): Unet_down(
            (down): down(
              (max_pool): MaxPool2d(kernel_size=2, stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
            )
            (conv): double_conv(
              (conv1): Conv2d(1

In [7]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.99, lr=0.0001)

In [8]:
from datetime import datetime

In [9]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('/tmp/kadota_runs/' +  datetime.now().strftime("%Y%m%d-%H%M%S"))

In [10]:
def train(model, loss_function, optimizer, dataloader, epoch=1):
    cur_loss = 0
    current_index = 0
    try:
        for e in range(epoch):
            for data in dataloader:

                sampled = data['sampled']
                mask = data['mask']
                undersampled = data['undersampled']
                for i in range(sampled.shape[0]):
                    optimizer.zero_grad()
                    sampled_slice = sampled[[i],...]
                    mask_slice = mask[[i],...]
                    undersampled_slice = undersampled[[i],...]
                    mask_slice = mask_slice.to(device)
                    undersampled_slice = undersampled_slice.to(device)
                    sampled_slice = sampled_slice.to(device)

                    predicted_sampled = model(undersampled_slice, mask_slice)
                    loss = loss_function(torch.view_as_real(predicted_sampled), torch.view_as_real(sampled_slice))

                    loss.backward()
                    optimizer.step()
                    cur_loss += loss.item()
                    writer.add_histogram('sens/weights1', next(model.sens_model.model.conv1d.parameters()), current_index)
                    writer.add_histogram('castcade0/weights1', next(model.cascade[0].unet.conv1d.parameters()), current_index)
                    writer.add_histogram('castcade5/weights1', next(model.cascade[5].unet.conv1d.parameters()), current_index)
                    writer.add_scalar('Loss/train', loss.item(), current_index)
                    if current_index % 10 == 9:
                        print(f"Iteration: {current_index + 1:>d}, Loss: {cur_loss:>7f}")
                        cur_loss = 0
                    current_index += 1
    except KeyboardInterrupt:
        pass

    model_name = model.__class__.__name__
    date = datetime.now().strftime("%Y%m%d-%H%M%S")
    torch.save(model.state_dict(), './Model_Weights/' + date + model_name)

In [11]:
model.cascade[0].unet.conv1d

Conv2d(18, 2, kernel_size=(1, 1), stride=(1, 1), bias=False)

In [12]:
train(model, loss_fn, optimizer, dataloader, epoch=2)

Iteration: 10, Loss: 6.252057
Iteration: 20, Loss: 4.457519
Iteration: 30, Loss: 4.311909
Iteration: 40, Loss: 4.791704
Iteration: 50, Loss: 2.701001
Iteration: 60, Loss: 0.529005
Iteration: 70, Loss: 3.785047
Iteration: 80, Loss: 4.126825
Iteration: 90, Loss: 4.528557
Iteration: 100, Loss: 1.488321
Iteration: 110, Loss: 0.298492
Iteration: 120, Loss: 0.310586
Iteration: 130, Loss: 0.251355
Iteration: 140, Loss: 0.298992
Iteration: 150, Loss: 3.699270
Iteration: 160, Loss: 4.519889
Iteration: 170, Loss: 4.888693
Iteration: 180, Loss: 2.103282
Iteration: 190, Loss: 1.264352
Iteration: 200, Loss: 0.377003
Iteration: 210, Loss: 2.155225
Iteration: 220, Loss: 4.756525
Iteration: 230, Loss: 4.552215
Iteration: 240, Loss: 2.370377
Iteration: 250, Loss: 0.259698
Iteration: 260, Loss: 3.922987
Iteration: 270, Loss: 4.883108
Iteration: 280, Loss: 4.153393
Iteration: 290, Loss: 4.110836
Iteration: 300, Loss: 3.918532
Iteration: 310, Loss: 4.592877
Iteration: 320, Loss: 4.248550
Iteration: 330, L